In [1]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain.tools import tool
from langchain.agents import create_agent

load_dotenv()

True

In [2]:
PRODUCTS = {
    "wireless headphones": {"price": 79.99,  "rating": 4.6, "description": "Over-ear Bluetooth, 30-hr battery, active noise cancellation."},
    "smart watch":         {"price": 199.99, "rating": 4.3, "description": "Tracks heart rate and sleep. 5-day battery, water-resistant."},
    "mechanical keyboard": {"price": 129.00, "rating": 4.8, "description": "Tenkeyless, Cherry MX Brown switches, per-key RGB."},
    "laptop stand":        {"price": 34.99,  "rating": 4.5, "description": "Adjustable aluminium, fits 11-17 inch laptops, folds flat."},
}

@tool
def get_product_info(product_name: str) -> str:
    """
    Get product information from the PRODUCTS dictionary return its price, rating, stock, and description.
    """
    product = PRODUCTS.get(product_name.lower())
    if product:
        return f"Product: {product_name}\nPrice: ${product['price']}\nRating: {product['rating']} stars\nDescription: {product['description']}"
    else:
        return f"Sorry, we don't have information on '{product_name}'."

In [3]:
# llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

agent = create_agent(
    llm,
    tools=[get_product_info],
    system_prompt="You are a helpful product assistant for an online tech store.",
)

In [4]:
def ask(question: str):
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    print(result["messages"][-1].content)

In [5]:
ask("what is the price of wireless headphones.")

The price of the wireless headphones is $79.99.


In [6]:
ask("tell me about mechanical keyboard")

The mechanical keyboard is a high-quality product with a price of $129.0. It has a rating of 4.8 stars, indicating excellent customer satisfaction. The keyboard features Cherry MX Brown switches, which are known for their tactile and audible feedback, and per-key RGB lighting, allowing for customizable backlighting. Additionally, it is a tenkeyless design, making it more compact and ideal for gamers or those who prefer a more minimalist setup.


In [7]:
REVIEWS = {
    "wireless headphones": {"reviews": 1262, "rating": 4.6},
    "smart watch":         {"reviews": 340,  "rating": 3.9},
    "mechanical keyboard": {"reviews": 67,   "rating": 4.8},
    "laptop stand":        {"reviews": 781,  "rating": 4.5},
}

@tool
def get_product_reviews(product_name: str) -> str:
    """"
    Get product reviews from the REVIEWS dictionary and return the number of reviews and average rarting."""
    product = REVIEWS.get(product_name.lower())
    if product:
        return f"Product: {product_name}\nNumber of Reviews: {product['reviews']}\nAverage Rating: {product['rating']} stars"
    else:
        return f"Sorry, we don't have review information on '{product_name}'."

In [10]:
# llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

agent2 = create_agent(
    llm,
    tools=[get_product_info, get_product_reviews],
    system_prompt="You are a helpful product assistant for an online tech store.",
)

def ask2(question: str):
    result = agent2.invoke({"messages": [{"role": "user", "content": question}]})
    print(result["messages"][-1].content)

In [9]:
ask2("how do people like smart watch")

{'messages': [HumanMessage(content='how do people like smart watch', additional_kwargs={}, response_metadata={}, id='baaa7bdf-25a0-4051-8e92-92096ac125f4'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'kvyeq2j3w', 'function': {'arguments': '{"product_name":"smart watch"}', 'name': 'get_product_reviews'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 342, 'total_tokens': 359, 'completion_time': 0.076681249, 'completion_tokens_details': None, 'prompt_time': 0.019171689, 'prompt_tokens_details': None, 'queue_time': 0.040635062, 'total_time': 0.095852938}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_f8b414701e', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a00ca4-b145-7043-a127-10842e528eb8-0', tool_calls=[{'name': 'get_product_reviews', 'args': {'product_name': 'smart watch'}, 'id': 'kvyeq2j3w', 'type': 'tool_call'}], inv

In [17]:
ask2("what are the reviews?")

I can't give you information on the reviews without knowing the specific product you're asking about. If you provide a product name, I can try to get the reviews for you.


In [11]:
ask2("what is the price and reviews of smart watch")

The price of the smart watch is $199.99. It has a rating of 4.3 stars and the description is that it tracks heart rate and sleep, has a 5-day battery, and is water-resistant. The smart watch has 340 reviews with an average rating of 3.9 stars.


In [14]:
from langgraph.checkpoint.memory import InMemorySaver
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

agent3_memory = create_agent(
    llm,
    tools=[get_product_info, get_product_reviews],
    system_prompt="You are a helpful product assistant for an online tech store.",
    checkpointer=InMemorySaver()
)

def ask3(question: str):
    config = {"configurable": {"thread_id": "user-alice-session-1"}}
    result = agent3_memory.invoke({"messages": [{"role": "user", "content": question}]}, config=config)
    print(result["messages"][-1].content)


In [15]:
ask3("what is the price of wireless headphones.")

The price of the wireless headphones is $79.99.


In [16]:
ask3("what are the reviews?")

The wireless headphones have 1262 reviews with an average rating of 4.6 stars.
